In [32]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/animatronbot/mnist-digit-recognizer/train.csv
/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_test.csv
/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv


In [ ]:
data = np.loadtxt('/kaggle/input/datasets/oddrationale/mnist-in-csv/mnist_train.csv', delimiter=",", skiprows=1)
np.random.shuffle(data)
split = int(0.8 * data.shape[0])
train_df = data[:split]
test_df = data[split:]
print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')

Train shape: (60000, 785)
Test shape:  (10000, 785)


In [34]:
def init_params():
    W1 = 0.1 * np.random.randn(128, 784)
    b1 = 0.1 * np.random.randn(128, 1)
    W2 = 0.1 * np.random.randn(10, 128)
    b2 = 0.1 * np.random.randn(10, 1)
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    return np.exp(Z) / np.sum(np.exp(Z), axis=0, keepdims=True)

In [35]:
def forward_pass(X, W1, b1, W2, b2):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

In [36]:
def back_prop(X, Y, W2, Z1, A1, Z2, A2, m):
    dZ2 = A2 - Y
    dW2 = dZ2.dot(A1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dA1 = W2.T.dot(dZ2)
    dZ1 = dA1 * (Z1 > 0)
    dW1 = dZ1.dot(X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    return dZ2, dW2, db2, dA1, dZ1, dW1, db1

In [37]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - (alpha * dW1)
    b1 = b1 - (alpha * db1)
    W2 = W2 - (alpha * dW2)
    b2 = b2 - (alpha * db2)
    return W1, b1, W2, b2

In [38]:
def train(X, Y, epochs, alpha):
    W1, b1, W2, b2 = init_params()
    m = Y.shape[1]

    for i in range(epochs):
        # Forward pass
        Z1, A1, Z2, A2 = forward_pass(X, W1, b1, W2, b2)

        # Backward pass
        dZ2, dW2, db2, dA1, dZ1, dW1, db1 = back_prop(
            X, Y, W2, Z1, A1, Z2, A2, m
        )

        # Update parameters
        W1, b1, W2, b2 = update_params(
            W1, b1, W2, b2, dW1, db1, dW2, db2, alpha
        )

        # Print progress every 50 epochs
        if i % 50 == 0:
            loss = np.mean(np.abs(dZ2))
            print(f"Epoch {i:>4} | Loss: {loss:.4f}")

    return W1, b1, W2, b2

In [39]:
def predict(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_pass(X, W1, b1, W2, b2)
    return np.argmax(A2, axis=0)

In [40]:
def accuracy(X, Y_labels, W1, b1, W2, b2):
    predictions = predict(X, W1, b1, W2, b2)
    return np.mean(predictions == Y_labels)

In [41]:
# Split into features and labels
X = data[:, 1:].T / 255.0     # (784, m) normalized to 0-1
Y_labels = data[:, 0].astype(int)  # (m,) integer labels

    # One-hot encode labels → (10, m)
m = Y_labels.shape[0]
Y = np.zeros((10, m))
Y[Y_labels, np.arange(m)] = 1

    # Train
W1, b1, W2, b2 = train(X, Y, epochs=500, alpha=0.1)

    # Evaluate
acc = accuracy(X, Y_labels, W1, b1, W2, b2)
print(f"\nFinal Training Accuracy: {acc * 100:.2f}%")

Epoch    0 | Loss: 0.1825
Epoch   50 | Loss: 0.0805
Epoch  100 | Loss: 0.0561
Epoch  150 | Loss: 0.0467
Epoch  200 | Loss: 0.0416
Epoch  250 | Loss: 0.0383
Epoch  300 | Loss: 0.0359
Epoch  350 | Loss: 0.0340
Epoch  400 | Loss: 0.0326
Epoch  450 | Loss: 0.0313

Final Training Accuracy: 91.83%
